# 2D Double-Slit Interference

**Phase 6** of `TDSE_Solver_Plan.md`: a Gaussian wavepacket incident on a wall
pierced by two slits (`potentials.double_slit_barrier`), producing the classic
interference pattern on the far side. The measured fringe spacing near the
pattern's center is compared to the analytic small-angle formula
`dy = lambda*L/d`, where `lambda=2*pi/k0` is the incident de Broglie wavelength,
`L` the distance from the wall to the measurement screen, and `d` the slit
separation.

That formula is a small-angle (paraxial) approximation -- exact fringe positions
satisfy `d*sin(theta_m) = m*lambda`, which is only close to linear in `y` near
`theta=0`. So fringe spacing is expected to match well near the pattern's
center and to visibly grow further out; both were checked directly (varying the
measurement window) before finalizing the numbers below, and the growth away
from center is the expected paraxial breakdown, not a bug.

In [1]:
import sys
sys.path.insert(0, r".")
import numpy as np
from scipy.signal import find_peaks
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import imageio_ffmpeg
from pathlib import Path

import grid as g
import propagator as prop
import potentials as pot
import observables as obs

matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
MEDIA_DIR = Path('media')
MEDIA_DIR.mkdir(exist_ok=True)


## Setup

Wall (thickness 2, height `V0=5`) at `x` in `[-1, 1]`, two slits of width 3
centered at `y=+-8` (separation `d=16`). Incident wavepacket: `k0=3`
(`lambda=2*pi/3~=2.09`) with a narrow `sigma_x=5` (well-defined propagation
energy, as in Phase 5) but a wide `sigma_y=20` so it illuminates both slits
almost uniformly (amplitude at `y=8` is ~96% of the peak). No absorbing
boundary is needed here -- the domain is comfortably larger than the distance
the packet travels in the time it takes to reach the screen, so nothing wraps
around.

In [2]:
Lx, Nx = 120.0, 384
Ly, Ny = 160.0, 512
grid2d = g.make_grid((Lx, Ly), (Nx, Ny), boundary='periodic')

k0 = 3.0
lam = 2 * np.pi / k0
d = 16.0
w = 3.0
V0 = 5.0
x_wall_lo, x_wall_hi = -1.0, 1.0
V = pot.double_slit_barrier(grid2d, V0, x_wall_lo, x_wall_hi, w, d)

sigma_x, sigma_y = 5.0, 20.0
x0, y0 = -30.0, 0.0
psi0 = g.gaussian_wavepacket(grid2d, center=(x0, y0), sigma=(sigma_x, sigma_y), k0=(k0, 0.0))

x_screen = 25.0
L_screen = x_screen - (x_wall_lo + x_wall_hi) / 2
T = (x_screen - x0) / k0
dt = 0.01
n_steps = round(T / dt)
print(f"lambda={lam:.4f}  L_screen={L_screen}  T={T:.2f}  n_steps={n_steps}")


lambda=2.0944  L_screen=25.0  T=18.33  n_steps=1833


In [3]:
target_frames = 150
stride = max(1, n_steps // target_frames)

k2 = prop.kinetic_eigenvalues(grid2d)
psi = psi0.copy()
frames = []
t = 0.0
for step in range(n_steps):
    psi = prop.strang_step(psi, grid2d, V, dt, k2=k2)
    t += dt
    if step % stride == 0:
        frames.append((t, obs.probability_density(psi).copy()))
frames.append((t, obs.probability_density(psi).copy()))
print(f"{len(frames)} frames recorded; final norm={obs.norm(psi, grid2d):.6f}")


154 frames recorded; final norm=0.999937


## Fringe spacing vs. the analytic formula

The screen profile is `|psi(x_screen, y)|^2`; peaks are found with
`scipy.signal.find_peaks` inside a window around the center, since (as
established above) spacing away from center isn't expected to match the
small-angle formula as closely.

In [4]:
density = obs.probability_density(psi)
x_axis, y_axis = grid2d.axes
i_screen = np.argmin(np.abs(x_axis - x_screen))
profile = density[i_screen, :]

analytic_spacing = lam * L_screen / d

half_window = 10.0
mask = np.abs(y_axis) <= half_window
y_win, p_win = y_axis[mask], profile[mask]
peaks, _ = find_peaks(p_win, prominence=p_win.max() * 0.15)
peak_y = y_win[peaks]
spacings = np.diff(peak_y)
measured_spacing = np.mean(spacings)

print(f"peaks found at y = {np.round(peak_y, 2)}")
print(f"measured mean spacing = {measured_spacing:.4f}")
print(f"analytic  spacing     = {analytic_spacing:.4f}")
rel_err = abs(measured_spacing - analytic_spacing) / analytic_spacing
print(f"relative error = {rel_err:.3f}")
assert rel_err < 0.05, "measured fringe spacing disagrees with the small-angle analytic formula by more than expected"
print("PASS: measured fringe spacing matches the small-angle analytic formula near the pattern center")


peaks found at y = [-6.56 -3.12  0.    3.12  6.56]
measured mean spacing = 3.2812
analytic  spacing     = 3.2725
relative error = 0.003
PASS: measured fringe spacing matches the small-angle analytic formula near the pattern center


In [5]:
# Also show how spacing grows away from center -- the expected paraxial
# breakdown, not a bug, and worth seeing explicitly rather than only
# checking the one narrow window above.
print(f"{'half-window':>12} {'mean spacing':>13} {'rel err vs analytic':>20}")
for hw in (10.0, 15.0, 18.0, 22.0):
    m = np.abs(y_axis) <= hw
    yw, pw = y_axis[m], profile[m]
    pk, _ = find_peaks(pw, prominence=pw.max() * 0.15)
    py = yw[pk]
    if len(py) > 1:
        sp = np.mean(np.diff(py))
        print(f"{hw:12.1f} {sp:13.4f} {abs(sp - analytic_spacing) / analytic_spacing:20.3f}")


 half-window  mean spacing  rel err vs analytic
        10.0        3.2812                0.003
        15.0        3.4375                0.050
        18.0        3.4375                0.050
        22.0        3.8750                0.184


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [2, 1]})
im = axes[0].imshow(density.T, origin='lower', extent=[x_axis[0], x_axis[-1], y_axis[0], y_axis[-1]],
                     cmap='inferno', aspect='auto', vmax=np.percentile(density, 99.9))
axes[0].axvspan(x_wall_lo, x_wall_hi, color='cyan', alpha=0.25)
axes[0].axvline(x_screen, color='lime', linestyle='--', linewidth=1)
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].set_title('final |psi|^2')

axes[1].plot(profile, y_axis, color='C0')
axes[1].plot(p_win[peaks], peak_y, 'o', color='C3', ms=5)
axes[1].set_ylim(y_axis[0], y_axis[-1])
axes[1].set_xlabel('|psi|^2 at screen')
axes[1].set_title(f'screen profile (x={x_screen})')
fig.suptitle(f'Double-slit: measured fringe spacing {measured_spacing:.3f} vs analytic {analytic_spacing:.3f}')
fig.tight_layout()
fig.savefig(MEDIA_DIR / 'double_slit_pattern.png', dpi=150)
plt.show()


C:\Users\Hasan's Laptop\AppData\Local\Temp\ipykernel_32796\2723845238.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
fig, ax = plt.subplots(figsize=(8, 6))
vmax = np.percentile(frames[len(frames) // 2][1], 99.9)
im = ax.imshow(frames[0][1].T, origin='lower', extent=[x_axis[0], x_axis[-1], y_axis[0], y_axis[-1]],
               cmap='inferno', aspect='auto', vmin=0, vmax=vmax)
ax.axvspan(x_wall_lo, x_wall_hi, color='cyan', alpha=0.25)
title = ax.set_title(f't=0.00')
ax.set_xlabel('x'); ax.set_ylabel('y')

def update(i):
    t_i, dens = frames[i]
    im.set_array(dens.T)
    title.set_text(f't={t_i:.2f}')
    return im, title

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=50, blit=False)
ani.save(MEDIA_DIR / 'double_slit_2d.mp4', writer='ffmpeg', fps=24, dpi=120)
plt.close(fig)
print('saved media/double_slit_2d.mp4')


saved media/double_slit_2d.mp4
